# Manual high-K coil characterisation

A standalone, short-pump-window notebook. It measures a trustworthy open-loop Q, proves the physical IN1â†’OUT2 spectrum-analyser path, locally refines feedback phase, calibrates K/A at two gains, and then replays manually selected K values. It intentionally omits Step 3b, drift campaigns, full phase/gain maps and high-density ladders.

## Wiring

```text
OUT1 -> 8 dB pad -> ZFL-500LN+ -> terminated tee/50 ohm -> coil -> IN1
OUT2 -> spectrum analyser (unity copy of IN1 during replay only)
```


---
# Configuration

> ### âœ… PUMP ON is fine
> Estimated bench time: **under 1 minute**. No hardware is driven by this cell. Review every field marked as a seed or installed-hardware value.


In [ ]:
import numpy as np
# ============================ CONFIGURATION ==================================
# This is the only cell intended for routine editing.
#
# SEARCH SEEDS: only locate the line; Step 2 replaces both with measurements.
F0_SEED_HZ = 39.9e6
Q_SEED = 210.0
PHASE_SEED_DEG = 0.0       # REQUIRED: recent approximate narrowing phase [deg]
PHASE_LOCAL_OFFSETS = (-30.0, -15.0, 0.0, 15.0, 30.0)

# INSTALLED HARDWARE: change whenever the physical drive chain changes.
EXPECTED_BUILD_ID = 0xFE76AE09
EXT_PAD_DB = 8.0
EXT_GAIN_DB = 28.2
EXT_GAIN_SAFETY_DB = 33.7
AMP_IN_MAX_DBM = 5.0
AMP_OUT_MAX_DBM = 0.0
AMP_P1DB_DBM = 7.9
DRIVE_FRAC = 0.0032
# Derived from +10 dBm DAC full scale, pad and worst-case gain.
DRIVE_MAX = 10.0 ** ((AMP_P1DB_DBM - 10.0 + EXT_PAD_DB - EXT_GAIN_SAFETY_DB)/20.0)
DRIVE_LINEAR_MAX = 10.0 ** ((AMP_OUT_MAX_DBM - 10.0 + EXT_PAD_DB - EXT_GAIN_SAFETY_DB)/20.0)

# RECEIVER/FIT: copied from the proven 4 K characterisation.
FS_CLK = 125e6
DEC_SEARCH = 8
DEC_FIT = 10
N_AVG = 64
FB_NAVG = 96
SETTLE = 0.02
GUARD_NSAMP = 256
LEVEL_GUARD_NSAMP = 64
LEVEL_GUARD_MODE = 'adaptive'
LEVEL_GUARD_EVERY = 8
LEVEL_GUARD_MARGIN = 0.35
LEVEL_GUARD_RISE = 4.0
FB_LEVEL_GUARD = 0.80
HIGH_Q_POINTS_PER_LW = 25
HIGH_Q_WING_LW = 4.0
HIGH_Q_RESONATOR_TAU = 7.0
FIR_TAPS = 309
FIR_SETTLE_GROUP_DELAYS = 4.0
SETTLING_CONVERGENCE_TOL = 0.01
FIT_TRUST_TOL = 0.02
SEARCH_FIT_MAX_RESID = 0.10
SEARCH_MIN_CORR = 0.50
SUBTRACT_BACKGROUND = True
FIT_ALLOW_SECOND = True
FIT_SECOND_F = 20.0
FIT_SECOND_TSTAT = 5.0
FIT_EXCLUDE_BANDS = []
FIT_ASYMMETRY_WARN = 0.08
COMPLEX_FIT_DIAGNOSTIC = False
HIGH_Q_COMPLEX_WIDTH_TOL = 0.05
SWEEP_SHUFFLE = True
FIR_CORRECT = True

# SHORT FEEDBACK CALIBRATION.
PHASE_PROBE_GAIN = 0.65
CAL_GAINS = (0.30, 0.50)
K_PER_A_LINEARITY_TOL = 0.10
FB_GAIN_MAX = 1.90
MANUAL_K_MAX = 0.95
MANUAL_K_STEP = 0.02
K_LADDER_MIN_COUNTS = 4

# SPECTRUM ANALYSER: max hold, logarithmic power.
OUT2_MON_SHIFT = 0         # unity; do not raise until unity is proven unclipped
OUT2_REPEATS = 3
OUT2_DWELL = 0.10
OUT2_ANALOGUE_CONFIRMED = False
OUT2_MONITOR_INPUT = False  # enabled explicitly only during guarded replay

# Compact emulator check. This validates the receiver/fitter, not the coil.
SELFTEST_F0 = F0_SEED_HZ
SELFTEST_Q = Q_SEED
SELFTEST_DRIVE = min(0.002, 0.5*DRIVE_LINEAR_MAX)
HIGH_Q_TEST_CENTRES_HZ = (F0_SEED_HZ,)
HIGH_Q_TEST_VALUES = (Q_SEED,)
HIGH_Q_TEST_DECIMATIONS = (DEC_FIT,)

assert np.isfinite(PHASE_SEED_DEG), 'PHASE_SEED_DEG must be a numeric recent estimate'
assert 0 < DRIVE_FRAC <= DRIVE_LINEAR_MAX <= DRIVE_MAX
print('configuration accepted')
print('  hard drive ceiling %.5f; conservative linear ceiling %.5f' %
      (DRIVE_MAX, DRIVE_LINEAR_MAX))


---
# Pump protocol and definitions

> ### âœ… PUMP ON is fine
> Estimated bench time: **under 1 minute**. These cells define and initialise the instrument with every output forced safe.


In [ ]:
import os, mmap, struct, time, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt


def s32(v):
    """Reinterpret a 32-bit register value as a signed integer."""
    return v - (1 << 32) if v & 0x80000000 else v


class Bus(object):
    """One register block, accessed strictly one 32-bit word at a time."""

    def __init__(self, base, span=0x1000):
        self.base = base
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m = mmap.mmap(self.fd, span, offset=base)

    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

    def rs(self, off):
        return s32(self.r(off))


# =============================================================================
#  PLOTTING COMPATIBILITY  -  this board runs matplotlib 1.x
# =============================================================================
#  The 'C0'..'C9' colour shorthand arrived in matplotlib 2.0. On 1.x it is not
#  rejected when you pass it - it is accepted, stored, and then raises
#
#      ValueError: to_rgba: Invalid rgba arg "C3"
#
#  from inside the draw, at which point the figure is already built. What you see
#  is a traceback followed by a bare <matplotlib.figure.Figure at 0x...> instead
#  of a plot, and every OTHER cell still runs, so it looks like a plotting
#  nuisance rather than a version problem. Every colour in this notebook is
#  therefore written out in full.
CLR_BLUE   = '#1f77b4'
CLR_ORANGE = '#ff7f0e'
CLR_GREEN  = '#2ca02c'
CLR_RED    = '#d62728'
CLR_GREY   = '0.5'

# The paper's figures use the modern perceptual colour maps; matplotlib 1.x has
# none of them, so viridis is rebuilt here from its own anchors - identical
# output on any version.
VIRIDIS_ANCHORS = [
    (0.2670, 0.0049, 0.3294), (0.2823, 0.0950, 0.4173), (0.2788, 0.1755, 0.4834),
    (0.2590, 0.2515, 0.5247), (0.2297, 0.3224, 0.5457), (0.1994, 0.3876, 0.5546),
    (0.1727, 0.4488, 0.5579), (0.1490, 0.5081, 0.5573), (0.1276, 0.5669, 0.5506),
    (0.1206, 0.6258, 0.5335), (0.1579, 0.6838, 0.5017), (0.2461, 0.7389, 0.4520),
    (0.3692, 0.7889, 0.3829), (0.5160, 0.8312, 0.2943), (0.6785, 0.8637, 0.1895),
    (0.8456, 0.8873, 0.0997), (0.9932, 0.9062, 0.1439)]
CMAP_SEQ = matplotlib.colors.LinearSegmentedColormap.from_list(
    'viridis_local', VIRIDIS_ANCHORS, N=256)
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    'coolwarm_local',
    [(0.230, 0.299, 0.754), (0.553, 0.690, 0.996), (0.866, 0.866, 0.866),
     (0.968, 0.718, 0.612), (0.706, 0.016, 0.150)], N=256)
CMAP_SEQ.set_bad('black')
CMAP_DIV.set_bad('black')

# One place for the look of every figure. Setting it here means no cell needs to
# repeat grid, font and layout calls.
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.autolayout': True,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'lines.markersize': 4,
})

try:                       # inline figures when running under Jupyter...
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:          # ...and a file-only backend when running as a script
    matplotlib.use('Agg')

print('numpy %s, matplotlib %s' % (np.__version__, matplotlib.__version__))
if int(matplotlib.__version__.split('.')[0]) < 2:
    print('  matplotlib 1.x detected - using explicit hex colours (see above)')

PUMP_LOG = []
def pump(state, step, minutes):
    state = str(state).lower()
    if state not in ('on', 'off'):
        raise ValueError("pump state must be 'on' or 'off'")
    PUMP_LOG.append({'state': state, 'step': str(step), 'minutes': float(minutes),
                     'time': time.time()})
    print('='*68)
    print('%s  |  PUMP %s  |  expected %.1f min' % (step, state.upper(), minutes))
    print('='*68)


---
# Embedded hardware blocks

> ### ✅ PUMP ON is fine
> Estimated bench time: **under 1 minute**. Definitions initialise every block with drive, feedback, emulator and OUT2 safe.

These are copied from the proven coil-characterisation notebook. No project file or previous notebook is imported.

In [ ]:
# =============================================================================
#  ACQUISITION SYSTEM - base 0x40600000
# =============================================================================
#  VOLTMETER: tune the local oscillator to the drive, read PEEK_I/PEEK_Q.
#  RECORDER : stream to memory for spectra.
#  ORDER MATTERS: disable -> write settings -> reset -> enable. The decimator
#  holds a running total that assumes a fixed R. Retuning the oscillator is one
#  register write and is safe at any time.
# =============================================================================
class Acq(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = None

    # ---- setup ----
    def start(self, dec_log2, src=0):
        """Bring the chain up at a given decimation. Safe to call repeatedly."""
        self.w(0x00, 0)                 # disable while settings change
        self.w(0x08, dec_log2)          # decimation R = 2**dec_log2
        self.w(0x0C, src)               # 0 = IN1, 1 = IN2
        self.w(0x00, 1 | 2)             # enable + soft reset (self-clearing)
        self.w(0x00, 1)                 # running
        self.dec = dec_log2

    def stop(self):
        self.w(0x00, 0)

    # ---- tuning ----
    def tune_word(self, finc):
        """Set the local oscillator by its raw 32-bit frequency word."""
        self.w(0x04, int(finc) & 0xFFFFFFFF)

    def tune(self, f):
        """Set the local oscillator to the nearest word to f hertz."""
        self.tune_word(int(round(f / FS_CLK * (1 << 32))))

    # ---- voltmeter ----
    def measure(self, navg=None):
        """Average navg in-phase/quadrature pairs; return one complex number.

        At DC both are steady, so this is coherent averaging: a real signal adds
        linearly while noise averages down as the square root of navg."""
        navg = N_AVG if navg is None else navg
        zi = 0.0
        zq = 0.0
        for _ in range(navg):
            zi += self.rs(0x18)
            zq += self.rs(0x1C)
        return complex(zi / navg, zq / navg)

    # ---- housekeeping ----
    def f_out(self):
        """Output sample rate at the current decimation [Hz]."""
        return FS_CLK / (1 << self.dec)

    def window_hz(self):
        """Half-width of the CLEAN part of the receiver window [Hz].

        The decimator rejects by better than 75 dB only over the middle fifth of
        its output band, so the usable half-width is 0.1 x the output rate."""
        return 0.10 * self.f_out()

    def build_id(self):
        return self.r(0x48)

    def status2(self):
        return self.r(0x38)


acq = Acq()
print('acquisition mapped. BUILD_ID = 0x%08X' % acq.build_id())

# =============================================================================
#  SIGNAL GENERATOR - base 0x40200000, stock Red Pitaya module
# =============================================================================
#  Replays a 16384-entry table; frequency is one register write.
#      f = step x 125e6 / 2**30      (about 0.116 Hz per step)
#  Registers: 0x04 amplitude (0x2000 = x1.0), 0x08 table size, 0x0C start,
#  0x10 step, table at 0x10000. Control 0x00: 0x11 free-run, 0xC0 held at zero.
# =============================================================================
class Generator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
        self._step = None

    def load_sine(self):
        """Write one cycle of a sine into the table. Takes a second or two:
        16384 single-word writes, and single-word is not negotiable here."""
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)     # table size, fixed point, inclusive
        self.w(0x0C, 0)                     # start replaying from entry 0

    def freq_word(self, f):
        """Nearest 30-bit frequency word to f hertz."""
        return int(round(f / FS_CLK * (1 << 30)))

    def set_word(self, step):
        """Set frequency by raw word; return the frequency actually produced."""
        self._step = int(step)
        self.w(0x10, self._step)
        return self._step * FS_CLK / float(1 << 30)

    def freq(self, f):
        """Set frequency in hertz; return the frequency actually produced."""
        return self.set_word(self.freq_word(f))

    def lo_word(self):
        """The 32-bit local-oscillator word that lands this signal EXACTLY at
        0 Hz. The generator counts in units of 2**-30 of the clock and the local
        oscillator in units of 2**-32, so the conversion is a factor of four and
        carries no rounding error at all."""
        return 4 * self._step

    def amp(self, frac):
        """Output amplitude as a fraction of full scale. CLAMPED, deliberately.

        The register is Q1.13 with 0x2000 = x1.0, and it happily accepts up to
        0x3FFF = x1.9999. The write below is masked with & 0x3FFF, so before this
        guard existed the mask turned mistakes into silent, WRONG drive levels:

            frac = -0.5  ->  0x3000  ->  x1.5   (a sign slip drives 50% HARDER)
            frac =  2.0  ->  0x0000  ->  x0.0   (silence, and no complaint)
            frac =  1.5  ->  0x3000  ->  x1.5   (clipped square wave, not a sine)

        Every one of those is silent on the bench. Anything outside
        [0, DRIVE_MAX] is now refused instead, so the only way to drive the coil
        harder is to raise DRIVE_MAX in the configuration cell on purpose."""
        frac = float(frac)
        if not np.isfinite(frac):
            raise ValueError('drive amplitude is not a finite number: %r' % frac)
        if frac < 0.0:
            raise ValueError(
                'negative drive amplitude (%.4f). A sign has no meaning here, and '
                'the register would silently wrap it to x%.4f.'
                % (frac, (int(round(frac * 8192)) & 0x3FFF) / 8192.0))
        if frac > DRIVE_MAX:
            raise ValueError(
                'drive %.4f exceeds DRIVE_MAX = %.4f. Nothing in this notebook '
                'may drive harder than the ceiling; raise it in the configuration '
                'cell if that is really what you want.' % (frac, DRIVE_MAX))
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)

    def _control_a_b_zero(self, a_control):
        """Write channel-A control while forcing generator channel B to zero.

        OUT2 is the phase-shifter monitor plus ASG channel B. A plain channel-A
        write clears B's control bits, so stale B state from another program could
        corrupt an external measurement. Bits 23:22 hold B at zero/reset."""
        self.w(0x24, 0)                    # B amplitude and DC offset both zero
        self.w(0x00, 0x00C00000 | int(a_control))

    def assert_b_zero(self):
        """Require channel B amplitude/offset zero and its engine held reset."""
        packed, control = self.r(0x24), self.r(0x00)
        if packed != 0 or ((control >> 22) & 3) != 3:
            raise RuntimeError('generator B is not zero: AMP/DC=0x%08X CTRL=0x%08X'
                               % (packed, control))
        return {'b_amp_dc': packed, 'control': control}

    def run(self):
        self._control_a_b_zero(0x40)       # reset the A read pointer
        self._control_a_b_zero(0x11)       # A free-runs; B remains forced zero

    def off(self):
        self.amp(0.0)
        self._control_a_b_zero(0xC0)       # both outputs held at zero/reset


gen = Generator()
gen.off()
gen.load_sine()
print('generator ready, table loaded, output off')

# =============================================================================
#  PHASE SHIFTER - base 0x40300000   (Step 8 only)
# =============================================================================
#  Output = A cos(phi) x I + A sin(phi) x Q, Q being the input delayed a quarter
#  period. Two corrections, both from the module header:
#   1. The quarter-period delay is under one clock at 40 MHz, so add whole signal
#      periods and pick the k whose fraction sits nearest 0 or 1.
#   2. Fixed THREE-sample latency (L = 3 since FIX 6; one sample is 115 deg at
#      40 MHz, so a stale value rotates the whole axis). Subtracted from the
#      request so the delivered phase is the requested one.
#  Both describe the DIGITAL path only. Step 8 finds its phases by measurement.
#  "Feedback off" is coefficients (0, 0), NOT the bypass bit.
# =============================================================================
class PhaseShift(Bus):

    LATENCY = 3      # datapath latency in samples (L = 3 since FIX 6; see above)

    def __init__(self):
        Bus.__init__(self, 0x40300000)
        self.f_ref = None

    def config_delay(self, f):
        """Programme the quarter-period delay line for frequency f."""
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000.0:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)       # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        if best is None:
            raise ValueError('no usable quarter-period delay at %.3f MHz' % (f / 1e6))
        self.w(0x08, best[1])                       # whole clocks
        self.w(0x0C, int(round(best[2] * 16384)))   # fraction, Q0.14
        self.w(0x10, 0)                             # bypass OFF
        self.f_ref = f
        return best[1], best[2]

    def set(self, A, phi_deg):
        """Apply gain/phase and prove the exact CI/CQ words were accepted."""
        if self.f_ref is None:
            raise RuntimeError('call config_delay() first')
        if not (0.0 <= A < 1.99):
            # A x 16384 must stay inside int16. At A = 2.0 the coefficient is
            # 32768, which wraps to -32768 and silently INVERTS the feedback.
            raise ValueError('gain must be in [0, 1.99)')
        phi = phi_deg - 360.0 * self.LATENCY * self.f_ref / FS_CLK
        ci = int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF
        cq = int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF
        self.w(0x00, ci); self.w(0x04, cq)
        if self.readback()['ci'] != ci or self.readback()['cq'] != cq:
            raise RuntimeError('phase/gain coefficient readback failed')
        return ci, cq

    def readback(self):
        """All state that can alter feedback or the OUT2 observation."""
        return {'ci': self.r(0x00) & 0xFFFF, 'cq': self.r(0x04) & 0xFFFF,
                'delay': self.r(0x08), 'delta': self.r(0x0C),
                'bypass': self.r(0x10) & 1, 'monitor': self.r(0x18) & 0x3F}

    def assert_held(self, expected):
        got = self.readback()
        changed = {k: (expected[k], got[k]) for k in expected if got[k] != expected[k]}
        if changed:
            raise RuntimeError('feedback/OUT2 settings changed during replay: %r' % changed)
        return got

    def off(self):
        self.w(0x00, 0)
        self.w(0x04, 0)

    def monitor(self, source='input', shift=0):
        """Route a signal to OUT2 and verify the register accepted it.

        source='input' observes the coil/ADC signal used by feedback; 'output'
        observes the phase-shifted feedback contribution. shift is a saturating
        left shift (0..7). Keep it at zero until the external level is known."""
        sources = {'off': 0, 'input': 1, 'output': 2}
        if source not in sources:
            raise ValueError("monitor source must be 'off', 'input' or 'output'")
        shift = int(shift)
        if not (0 <= shift <= 7):
            raise ValueError('monitor shift must be in [0, 7]')
        value = sources[source] | (shift << 3)
        self.w(0x18, value)
        got = self.r(0x18)
        if (got & 0x3F) != value:
            raise RuntimeError('OUT2 monitor readback failed: wrote 0x%02X, read 0x%08X; check the bitstream' % (value, got))
        return got

    def monitor_clipped(self):
        """True if OUT2 monitor scaling has clipped since configuration."""
        return bool(self.r(0x18) & (1 << 8))

    def monitor_off(self):
        self.monitor('off', 0)


ps = PhaseShift()
ps.off()
if OUT2_MONITOR_INPUT:
    ps.monitor('input', shift=0)
    print('phase shifter mapped, feedback off; OUT2 copies IN1 at unity')
else:
    ps.monitor_off()
    print('phase shifter mapped, feedback off; OUT2 disabled')

# =============================================================================
#  DIGITAL RESONATOR (THE EMULATOR) - base 0x40400000
# =============================================================================
#  Used only in Step 1. Enabled, it REPLACES the digitised input, so the coil can
#  stay connected and simply be ignored - a known answer is available at any
#  moment without recabling.
#  Coefficients c = r cos(w0) x 2**17, s = r sin(w0) x 2**17, w0 = 2 pi f0/125e6,
#  r = 1 - pi (f0/125e6)/Q. Rounding moves the pole, so the QUANTISED values
#  recovered below are the real known answer.
# =============================================================================
class Emulator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None
        self.q_q = None

    def config(self, f0, Q, delay=12, oshift=7):
        """Set the emulated resonance. FOUR coefficients, since 2026-08-07.

        The recursion advances two samples per update - two-way look-ahead, see
        the module header - so the pole it actually realises is the one in
        COEF_C2/COEF_S2: r squared, turned by 2*w0. COEF_C and COEF_S are still
        required, because they carry the drive term, so all four are written.

        The quantised answer returned therefore comes from c2 and s2, since that
        is the pole the hardware has. Comparing the Step 1 self-test against
        anything else would build a permanent error into its tolerance.

        `delay` is the EXTRA delay, on top of the module's own pipeline. That
        pipeline is three clocks since 2026-08-07 - one from the drive leg, one
        from the state output, one from the output-stage register added to close
        the last of the timing - so 12 + 3 = the board's real 15. It was 15 + 0
        before any of this, then 13 + 2. If a future build adds another stage,
        this number moves again, and Step 1 is what will notice."""
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        c2 = int(round(r * r * np.cos(2.0 * w0) * 131072))
        s2 = int(round(r * r * np.sin(2.0 * w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x28, c2 & 0x3FFFF)
        self.w(0x2C, s2 & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        # arctan2 gives 2*w0 modulo 2*pi, and 2*w0 passes pi above 31.25 MHz,
        # so lift it onto the branch nearest the frequency asked for - which is
        # known here, because it is what produced the coefficients.
        r2 = np.hypot(c2, s2) / 131072.0
        two_w0 = np.arctan2(s2, c2)
        two_w0 += 2.0 * np.pi * round((2.0 * w0 - two_w0) / (2.0 * np.pi))
        self.f0_q = (two_w0 / 2.0) / (2.0 * np.pi) * FS_CLK
        self.q_q = np.pi * (self.f0_q / FS_CLK) / (1.0 - np.sqrt(r2))
        #  Prove the write took. COEF_C2 exists only in the four-coefficient
        #  bitstream; an older one does not decode 0x28 and returns zero, and
        #  then the recursion has no pole at all - the emulator becomes a
        #  pass-through with a flat response, which a Lorentzian fitter turns
        #  into a plausible-looking wrong answer rather than an error.
        if ((self.r(0x28) & 0x3FFFF) != (c2 & 0x3FFFF) or
                (self.r(0x2C) & 0x3FFFF) != (s2 & 0x3FFFF)):
            raise RuntimeError(
                'COEF_C2/COEF_S2 did not read back. This notebook writes the '
                'four-coefficient emulator (two-way look-ahead, 2026-08-07); '
                'the bitstream on the board does not have it. Rebuild and '
                'reload. NOTE the opposite mistake is SILENT: an old notebook '
                'on new hardware leaves c2 = s2 = 0, the recursion has no pole, '
                'and Step 1 reports a fitted Q near 1 with a centre outside the '
                'swept band.')
        return self.f0_q, self.q_q

    def enable(self):
        self.w(0x00, 1)

    def disable(self):
        self.w(0x00, 0)

    def clear(self):
        self.w(0x00, 3)          # zero the state and the sticky clip flag

    def enabled(self):
        return bool(self.r(0x00) & 1)

    def clipped(self):
        return bool(self.r(0x14) & 1)


emu = Emulator()
emu.disable()
print('emulator mapped and DISABLED (the board is listening to its real input)')

# =============================================================================
#  INTERNAL SCOPE - base 0x40100000
# =============================================================================
#  16384 raw samples at up to 125 MSa/s. The only overload guard on the real
#  path: the emulator's sticky clip flag is gone once a real coil is measured.
#  Sees the same signal as the rest of the chain, so with the emulator enabled it
#  shows the emulator.
# =============================================================================
SC_BASE = 0x40100000
SC_SPAN = 0x40000
SC_BUF_A = 0x10000
SC_BUF_B = 0x20000
SC_NMAX = 1 << 14


class Scope(Bus):

    def __init__(self):
        Bus.__init__(self, SC_BASE, span=SC_SPAN)

    def _buf(self, off, n):
        """Read n samples, one 32-bit word at a time (this is a register block,
        not DDR). The buffer holds 14-bit two's-complement values."""
        ba = bytearray(4 * n)
        m = self.m
        for i in range(n):
            o = off + 4 * i
            ba[4 * i:4 * i + 4] = m[o:o + 4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=SC_NMAX, chan='A'):
        """Capture and return (samples, sample_rate). dec = 1 is every sample."""
        self.w(0x14, dec)
        self.w(0x10, n - 1)
        self.w(0x00, 2)        # reset the capture engine
        self.w(0x00, 1)        # arm
        self.w(0x04, 1)        # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        off = SC_BUF_A if chan == 'A' else SC_BUF_B
        return self._buf(off, n), FS_CLK / float(dec)

    def peak_frac(self, n=4096):
        """TOTAL peak excursion as a fraction of full scale. 1.0 means clipping.

        Deliberately includes everything - the converter's DC offset, broadband
        noise, out-of-band pickup - because all of it counts towards clipping.
        That makes it the right guard and the WRONG way to measure the size of a
        tone. Use tone_frac() for that."""
        x, _ = self.acquire(dec=1, n=n)
        return float(np.max(np.abs(x))) / 8191.0

    def tone_frac(self, f, n=4096):
        """Amplitude of the component at f alone, as a fraction of full scale.

        WHY THIS EXISTS. peak_frac() answers "how close am I to clipping". It was
        also being used to answer "how much 40 MHz is arriving", and those are
        not the same question. max|x| is inflated by the converter's DC offset
        and by every other thing in the record, so on this board it read about
        four times the true tone amplitude - which then over-predicted the
        feedback loop gain by the same factor.

        A least-squares projection onto cos and sin at the known frequency fixes
        it: the mean is removed first, so DC cannot contribute, and anything not
        at f is orthogonal to both basis vectors and averages away. No window and
        no scalloping loss, because f is known exactly rather than searched."""
        x, fs = self.acquire(dec=1, n=n)
        x = x - np.mean(x)                       # the converter's DC offset
        t = np.arange(len(x)) / fs
        basis = np.column_stack([np.cos(2.0 * np.pi * f * t),
                                 np.sin(2.0 * np.pi * f * t)])
        coef, _, _, _ = np.linalg.lstsq(basis, x, rcond=-1)
        return float(np.hypot(coef[0], coef[1])) / 8191.0


sc = Scope()
print('scope mapped, %d samples per capture' % SC_NMAX)


# =============================================================================
#  ALL_OFF  -  the one call that makes the hardware harmless
# =============================================================================
#  Every step below that raises the drive does so inside a try/finally that ends
#  here, so an exception, a failed fit or an interrupted cell cannot leave the
#  coil being driven with the feedback loop closed. It is also registered to run
#  when the kernel shuts down.
#
#  Safe to call at any time, including twice, including when nothing is running.
#  If anything ever looks wrong on the bench, run all_off() in a fresh cell.
# =============================================================================
def all_off():
    """Silence the drive, open the feedback loop, disable the emulator."""
    for what, fn in (('generator', lambda: gen.off()),
                     ('feedback', lambda: ps.off()),
                     ('OUT2 monitor', lambda: ps.monitor_off()),
                     ('acquisition', lambda: acq.stop()),
                     ('emulator', lambda: emu.disable())):
        try:
            fn()
        except Exception as exc:
            # One block failing to switch off must never stop the others being
            # switched off - the generator is the one that matters.
            print('all_off: could not switch off the %s (%s)' % (what, exc))


import atexit
atexit.register(all_off)
all_off()
if OUT2_MONITOR_INPUT:
    ps.monitor('input', shift=0)
print('all_off() defined and armed - drive, feedback and emulator are all off')
print('OUT2 monitor: %s' % ('IN1 copy at unity' if OUT2_MONITOR_INPUT else 'off'))

# =============================================================================
#  MEMORY CAPTURE
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None


_blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert _blob and len(_blob) >= 8, 'cannot read the device tree memory node'
_declared_end = _be32(_blob, 0) + _be32(_blob, 1)
assert _declared_end < 512 * 1024 ** 2, \
    'no memory above the declared end - see Test Stage 6 Step 1'

MEM_START = (_declared_end + 0xFFF) & ~0xFFF
MEM_SIZE = 0x00800000                        # 8 MB is far more than needed
MEM_STOP = MEM_START + (MEM_SIZE & ~7) - 8   # LAST address, INCLUSIVE
POISON = 0xDEADBEEF

print('capture region 0x%08X .. 0x%08X inclusive (%d MB)'
      % (MEM_START, MEM_STOP + 7, MEM_SIZE // (1024 * 1024)))


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((nbytes + 4095) // 4096) * 4096, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)


def _read_region(nbytes):
    """Bulk read of DDR. try/finally because everything here can raise, and a
    leaked mapping inside a scan loop surfaces much later as an unrelated-
    looking out-of-memory error."""
    fd = mm = None
    try:
        fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
        return mm[0:nbytes]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)


def capture(lo_word, dec_log2, n_samples, settle=None):
    """Record n_samples of the complex stream. Returns a complex numpy array.

    lo_word is the RAW 32-bit local-oscillator word, so a caller that wants an
    exact zero-frequency landing can pass gen.lo_word() and get it. Raises on
    any transport fault rather than returning plausible garbage."""
    settle = SETTLE if settle is None else settle
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples exceeds the capture region' % n_samples)

    _poison(n_samples * 8)

    acq.w(0x00, 0)                       # disable while settings change
    acq.w(0x08, dec_log2)
    acq.w(0x0C, 0)
    acq.dec = dec_log2
    acq.tune_word(lo_word)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)

    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded memory writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))

    acq.w(0x00, 1 | 2)                   # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                   # ...let the pipeline flush...
    acq.w(0x00, 1 | 16 | 128)            # ...then arm with a fresh writer
    acq.w(0x00, 1 | 16)

    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                       # disarm
    time.sleep(0.002)                    # let the last burst land

    st2 = acq.r(0x38)
    if st2 & 8:
        raise RuntimeError('memory refused writes - run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    if got < n_samples:
        raise RuntimeError('short record: asked %d, got %d' % (n_samples, got))

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    limit = max(8, 0.001 * u32.size)
    if int(np.count_nonzero(u32 == POISON)) > limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms.
        time.sleep(0.02)
        u32 = np.frombuffer(_read_region(got * 8), dtype='<u4')
        if int(np.count_nonzero(u32 == POISON)) > limit:
            raise RuntimeError('unwritten words in the record - transport fault')

    iq = u32.astype(np.int64)
    i = np.where(iq[0::2] & 0x80000000, iq[0::2] - (1 << 32), iq[0::2])
    q = np.where(iq[1::2] & 0x80000000, iq[1::2] - (1 << 32), iq[1::2])
    return i.astype(float) + 1j * q.astype(float)



---
# Embedded analysis

> ### ✅ PUMP ON is fine
> Estimated bench time: **under 10 seconds**. This cell defines analysis functions and does not drive hardware.

The robust constrained fitter, tracked zero-IF sweep, background subtraction and independent width diagnostics are the established characterisation implementations.

In [ ]:
# =============================================================================
#  ANALYSIS
# =============================================================================
#  Two anchors for the input-peak predictor the level guard uses: the lowest
#  and highest (|z|, measured peak) pairs seen so far. Affine between them and
#  extrapolated beyond, floored at the low anchor - so a prediction is never
#  smaller than the standing offset and noise the converter always shows.
_GUARD_CAL = [None, None]     # [(z_lo, pk_lo), (z_hi, pk_hi)]


def _guard_learn(z_abs, peak_frac_seen):
    lo, hi = _GUARD_CAL
    pair = (float(z_abs), float(peak_frac_seen))
    if lo is None or pair[0] < lo[0]:
        _GUARD_CAL[0] = pair
    if hi is None or pair[0] > hi[0]:
        _GUARD_CAL[1] = pair


def _guard_predict(z_abs):
    """Predicted input peak for a given |z|, or inf if not yet calibrated.

    Deliberately biased high, and refuses to answer at all until the two
    anchors are far enough apart to define a slope - a single anchor would
    predict a CONSTANT peak, which is exactly the wrong answer on the rising
    flank of a resonance."""
    lo, hi = _GUARD_CAL
    if lo is None or hi is None or hi[0] < 2.0 * max(lo[0], 1e-30):
        return float('inf')
    slope = (hi[1] - lo[1]) / (hi[0] - lo[0])
    return max(lo[1], lo[1] + slope * (float(z_abs) - lo[0]))


def guard_reset():
    """Forget the calibration.

    sweep() calls this on entry, every time. That is deliberate: the mapping
    from |z| to input peak depends on the drive, the decimation AND the
    feedback gain, and a calibration carried across a gain change could
    under-predict by 1/(1-K) - twenty-fold near K = 1. Learning it fresh inside
    each sweep costs one guaranteed check and cannot go stale."""
    _GUARD_CAL[0] = _GUARD_CAL[1] = None


def sweep(freqs, drive_frac=None, navg=None, settle=None, level_guard=None):
    """Drive at each frequency and read the complex response at DC.

    Returns (f_actual, z): f_actual is what the generator really produced (its
    word is quantised) and z is complex. Only |z| is a measurement - see Step 5.

    The receiver is tuned by WORD, not by hertz, so the signal lands at exactly
    zero frequency and the two register reads cannot land on different phases of
    a rotating vector."""
    drive_frac = DRIVE_FRAC if drive_frac is None else drive_frac
    navg = N_AVG if navg is None else navg
    settle = SETTLE if settle is None else settle

    gen.amp(drive_frac)                  # refused outright if above DRIVE_MAX
    gen.run()
    f_act = np.zeros(len(freqs))
    z = np.zeros(len(freqs), dtype=complex)
    mode = str(globals().get('LEVEL_GUARD_MODE', 'adaptive'))
    every = max(1, int(globals().get('LEVEL_GUARD_EVERY', 8)))
    margin = float(globals().get('LEVEL_GUARD_MARGIN', 0.35))
    rise = float(globals().get('LEVEL_GUARD_RISE', 4.0))
    checks = 0
    guard_reset()
    try:
        for k, f in enumerate(freqs):
            f_act[k] = gen.freq(f)
            acq.tune_word(gen.lo_word())
            time.sleep(settle)
            #  THE GUARD IS THE MOST EXPENSIVE THING IN THE NOTEBOOK.
            #  sc.peak_frac() arms the scope, sleeps 5 ms and then reads
            #  GUARD_NSAMP words one at a time through Python. On this board a
            #  register access costs about 180 us, so one guard check is ~50 ms
            #  against ~12 ms for the measurement it is protecting. Run on every
            #  point it was 67% of the Step 8c map's 16.5 minutes - more than the
            #  measurement, the settling and the oscillation checks combined.
            #
            #  It cannot simply be thinned, because the whole point is to catch
            #  the input running away. So instead: PREDICT the input peak from
            #  the |z| already being measured, and spend a real check only when
            #  the prediction comes near the limit. Two anchors, learned from
            #  real checks, give peak(|z|) as an affine law - affine and not a
            #  ratio because the converter's standing offset and noise do not
            #  scale with the tone. The prediction is inflated by `rise` so a
            #  response climbing steeply between points cannot outrun it, and a
            #  real check still happens on the first point and every `every`
            #  points regardless. Set LEVEL_GUARD_MODE = 'every' to restore the
            #  old behaviour exactly.
            if level_guard is not None:
                need = True
                if mode == 'adaptive' and k > 0 and (k % every) and _GUARD_CAL[0]:
                    need = (_guard_predict(rise * abs(z[k - 1]))
                            > margin * level_guard)
                if need:
                    pk = sc.peak_frac(LEVEL_GUARD_NSAMP)
                    checks += 1
                    if pk > level_guard:
                        raise RuntimeError(
                            'input exceeded %.0f%% of full scale at %.6f MHz'
                            % (100 * level_guard, f_act[k] / 1e6))
                else:
                    pk = None
            else:
                pk = None
            z[k] = acq.measure(navg)
            if pk is not None:
                _guard_learn(abs(z[k]), pk)
    finally:
        # Whatever ended this sweep - the guard above, a bad register read, a
        # Ctrl-C halfway through a 350-point pass - the drive comes off. Every
        # measurement goes through here, so no failure path can walk away
        # leaving the coil driven.
        gen.amp(0.0)
    globals()['LAST_SWEEP_GUARD_CHECKS'] = int(checks)
    globals()['LAST_SWEEP_POINTS'] = int(len(freqs))
    return f_act, z


def preflight_drive(f_probe, target=None):
    """Approach the working drive in two steps instead of jumping to it.

    THE FAILSAFE THAT MATTERS. Every other guard assumes EXT_GAIN_DB is right; if
    it is understated, DRIVE_MAX is too permissive and the first full-drive write
    is the one that does the damage. So measure at a tenth of the drive,
    extrapolate, and refuse before the converter ever sees the full value."""
    target = DRIVE_FRAC if target is None else target
    lo = max(target / 10.0, 2.0e-4)
    try:
        gen.freq(f_probe); gen.amp(lo); gen.run(); time.sleep(0.05)
        small = sc.peak_frac(GUARD_NSAMP)
    finally:
        gen.off()
    pred = small * target / lo
    print('preflight: %.4f of full scale at drive %.5f -> %.3f predicted at %.5f'
          % (small, lo, pred, target))
    if pred > 0.90:
        raise RuntimeError(
            'a drive of %.5f would put %.2f of full scale into the input. '
            'Something delivers far more than expected - check EXT_GAIN_DB '
            '(declared %.1f dB) and that the amplifier is in the DRIVE path, '
            'not the pickup path.' % (target, pred, EXT_GAIN_DB))
    if small > 0.60:
        raise RuntimeError('input already at %.2f of full scale at a tenth of '
                           'the working drive - stop and investigate.' % small)
    return pred


def _lorentz_model(f, t, n_extra=0, slope=False):
    """t = [f0, lw, A, b] (+ [s] if slope) (+ [f0k, lwk, Ak] per extra peak)."""
    m = t[2] / (1.0 + (2.0 * (f - t[0]) / t[1]) ** 2) + t[3]
    o = 4
    if slope:
        m = m + t[4] * (f - t[0]) / max(abs(t[1]), 1e-30)
        o = 5
    for k in range(n_extra):
        i = o + 3 * k
        m = m + t[i + 2] / (1.0 + (2.0 * (f - t[i]) / t[i + 1]) ** 2)
    return m


def _lorentz_jac(f, t, n_extra=0, slope=False):
    """Analytic derivatives. Written out rather than differenced because a
    finite difference on a peak this sharp is itself a source of bias."""
    J = np.zeros((len(f), len(t)))
    x = 2.0 * (f - t[0]) / t[1]
    d = 1.0 + x * x
    J[:, 0] = t[2] * 4.0 * x / (t[1] * d ** 2)
    J[:, 1] = t[2] * 2.0 * x * x / (t[1] * d ** 2)
    J[:, 2] = 1.0 / d
    J[:, 3] = 1.0
    o = 4
    if slope:
        J[:, 0] += -t[4] / max(abs(t[1]), 1e-30)
        J[:, 4] = (f - t[0]) / max(abs(t[1]), 1e-30)
        o = 5
    for k in range(n_extra):
        i = o + 3 * k
        xb = 2.0 * (f - t[i]) / t[i + 1]
        db = 1.0 + xb * xb
        J[:, i] = t[i + 2] * 4.0 * xb / (t[i + 1] * db ** 2)
        J[:, i + 1] = t[i + 2] * 2.0 * xb * xb / (t[i + 1] * db ** 2)
        J[:, i + 2] = 1.0 / db
    return J


def _levenberg(fun, jac, t0, w=None, maxiter=300, ftol=1e-14):
    """Levenberg-Marquardt in numpy alone - there is no scipy on the board."""
    t = np.array(t0, dtype=float)
    lam = 1e-3
    ww = np.ones(len(fun(t))) if w is None else np.asarray(w, dtype=float)
    r = fun(t) * ww
    cost = float(np.dot(r, r))
    JtJ = None
    for _ in range(maxiter):
        J = jac(t) * ww[:, None]
        JtJ = np.dot(J.T, J)
        g = np.dot(J.T, r)
        stepped = False
        for _inner in range(40):
            D = np.diag(np.maximum(np.diag(JtJ), 1e-30))
            try:
                step = np.linalg.solve(JtJ + lam * D, -g)
            except np.linalg.LinAlgError:
                lam *= 10.0
                continue
            tn = t + step
            rn = fun(tn) * ww
            cn = float(np.dot(rn, rn))
            if np.isfinite(cn) and cn < cost:
                done = abs(cost - cn) <= ftol * max(cost, 1e-300)
                t, r, cost = tn, rn, cn
                lam = max(lam * 0.3, 1e-12)
                stepped = True
                if done:
                    return t, cost, JtJ
                break
            lam *= 10.0
            if lam > 1e14:
                break
        if not stepped:
            break
    return t, cost, JtJ


def _lorentz_seed(f, p):
    """A starting point that cannot fail: edges for the floor, the largest
    point for the centre, the half-maximum span for the width."""
    order = np.argsort(f)
    f, p = f[order], p[order]
    edge = max(2, len(p) // 8)
    base = float(np.median(np.r_[p[:edge], p[-edge:]]))
    q = p - base
    A = float(np.max(q))
    f0 = float(f[int(np.argmax(q))])
    above = q > 0.5 * A
    lw = (float(np.max(f[above]) - np.min(f[above]))
          if int(np.count_nonzero(above)) >= 2 else float(np.ptp(f) / 10.0))
    lw = max(lw, 2.0 * float(np.median(np.abs(np.diff(f)))), 1.0)
    return [f0, lw, A, max(base, 0.0)]


def _legacy_linear_fit(f, p, guard=0.05, minpts=5):
    """The old reciprocal-parabola estimate. Kept ONLY as a starting guess.

    It is closed-form, so it always returns something and never fails to
    converge - exactly what a seed must do, and exactly what an ANSWER must not
    be allowed to get away with."""
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        k = q > guard * q.max()
        if int(np.count_nonzero(k)) < minpts:
            return None
        A, B, C = np.polyfit(u[k], 1.0 / q[k], 2, w=q[k])
        if A <= 0.0:
            return None
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        model = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - model) / pmax) ** 2))), (f0, lw, p0, b)

    best, best_b = _try(0.0), 0.0
    hi = 0.999 * float(np.min(p))
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('linearised seed failed')
    return list(best[1])


def fit_lorentzian_legacy(f, p, guard=0.05, minpts=5):
    """The fit this notebook used until the 4 K long-wire run. Kept so every
    plot can show, side by side, what the change is worth on real data."""
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    f0, lw, peak, base = _legacy_linear_fit(f, p, guard, minpts)
    model = peak / (1.0 + (2.0 * (f - f0) / lw) ** 2) + base
    resid = float(np.sqrt(np.mean(((p - model) / np.max(p)) ** 2)))
    return dict(f0=f0, lw=lw, peak=peak, base=base, resid=resid, Q=f0 / lw,
                n_components=1, second=None, model_t=np.array([f0, lw, peak, base]))


def _fit_once(f, p, t0, n_extra=0, slope=False, robust=True, irls=5,
              huber=2.5, weight='invsqrt'):
    """Weighted, optionally robust nonlinear least squares on POWER."""
    scale = max(float(np.max(p)), 1e-300)

    def fun(t):
        return (_lorentz_model(f, t, n_extra, slope) - p) / scale

    def jac(t):
        return _lorentz_jac(f, t, n_extra, slope) / scale

    t = np.array(t0, dtype=float)
    w = np.ones(len(f))
    JtJ = None
    for _ in range(max(1, irls)):
        t, cost, JtJ = _levenberg(fun, jac, t, w=w)
        t[1] = abs(t[1])
        t[3] = max(t[3], 0.0)
        for k in range(n_extra):
            i = (5 if slope else 4) + 3 * k
            t[i + 1] = abs(t[i + 1])
        m = _lorentz_model(f, t, n_extra, slope)
        r = (p - m) / scale
        #  |z|^2 of a tone in complex noise has a variance proportional to its
        #  own mean, so the natural weight is 1/sqrt(model). Uniform weighting
        #  lets the top of the line set the width on its own; 1/model hands the
        #  width to the wings, which is where the interference lives.
        w = (1.0 / np.sqrt(np.maximum(m, 1e-300) / scale)
             if weight == 'invsqrt' else np.ones(len(f)))
        if robust:
            rw = r * w
            s = 1.4826 * float(np.median(np.abs(rw - np.median(rw))))
            if s > 0.0:
                a = np.abs(rw) / s
                w = w * np.where(a <= huber, 1.0,
                                 np.sqrt(huber / np.maximum(a, 1e-300)))
        w = w / max(float(np.max(w)), 1e-300)
    m = _lorentz_model(f, t, n_extra, slope)
    resid = float(np.sqrt(np.mean(((p - m) / scale) ** 2)))
    return t, resid, JtJ, w


def _t_statistic(JtJ, resid, n, k, index, value):
    """|parameter| divided by its own standard error, from the fit covariance."""
    if JtJ is None or n <= k:
        return 0.0
    try:
        cov = np.linalg.inv(JtJ) * (resid ** 2) * n / float(n - k)
    except np.linalg.LinAlgError:
        return 0.0
    v = float(cov[index, index])
    return abs(value) / np.sqrt(v) if v > 0.0 else 0.0


def _seed_second_peak(f, p, t1, slope=False):
    """Where does the one-peak fit leave a positive, STRUCTURED excess?

    Smoothed, so a single high point cannot start a second component, and
    restricted to beyond 0.75 linewidths, so the seed can never be the line
    itself re-found."""
    r = p - _lorentz_model(f, t1, 0, slope)
    k = max(3, len(f) // 25)
    sm = np.convolve(r, np.ones(k) / float(k), mode='same')
    away = np.abs(f - t1[0]) > 0.75 * t1[1]
    if not np.any(away):
        return None
    j = int(np.argmax(np.where(away, sm, -np.inf)))
    if sm[j] <= 0.0:
        return None
    half = 0.5 * sm[j]
    lo, hi = f[j], f[j]
    for i in range(j, -1, -1):
        if sm[i] < half:
            break
        lo = f[i]
    for i in range(j, len(f)):
        if sm[i] < half:
            break
        hi = f[i]
    return [float(f[j]), float(max(hi - lo, 0.5 * t1[1])),
            float(max(sm[j], 1e-3 * t1[2]))]


def _flank_asymmetry(f, p, fit):
    """Fit each flank on its own. A clean line gives the same width twice.

    This is the MODEL-FREE alarm. It assumes nothing about what the interferer
    is - only that a Lorentzian is symmetric and a contaminated one is not - so
    it fires even in the cases the two-component model declines to take."""
    try:
        widths = []
        for side in (-1.0, 1.0):
            m = (side * (f - fit['f0']) >= -0.15 * fit['lw'])
            if int(np.count_nonzero(m)) < 8:
                return float('nan')
            t, _r, _J, _w = _fit_once(f[m], p[m], [fit['f0'], fit['lw'],
                                                   fit['peak'], fit['base']],
                                      0, False, False, irls=3)
            widths.append(abs(t[1]))
        return float(abs(widths[1] - widths[0]) /
                     max(0.5 * (widths[0] + widths[1]), 1e-30))
    except Exception:
        return float('nan')


def fit_lorentzian(f, p, guard=0.05, minpts=5, exclude=None, allow_second=None,
                   allow_slope=False, second_gain=0.92, second_f=None,
                   second_t=None, robust=True, diagnostics=True):
    """Centre, linewidth, peak and baseline from power p against frequency f.

    WHY THIS IS NO LONGER THE RECIPROCAL-PARABOLA FIT.  1/p is a parabola in f
    for a pure Lorentzian on a known floor, and that identity makes a closed-form
    fit possible. But it is an identity about the MODEL, not about the data.
    Anything additive that is not the line - a second mode, a standing wave on
    the several-metre wiring, a board spur the receiver does not fully reject -
    breaks it, and it breaks in ONE DIRECTION: the parabola flattens, so the
    linewidth comes out too large and Q too small, and the centre walks towards
    the interferer. On the 17%-of-peak shoulder this cooldown actually shows,
    the error is +86% in linewidth and 0.2 linewidths in centre. That is not a
    precision problem. It changes what the notebook concludes about the coil,
    and it does it silently, because the linearised fit's residual is the only
    thing that complains and a 0.09 residual still reads as "nearly fine".

    What replaces it:

      * a true nonlinear least-squares fit of the Lorentzian AND its floor,
        which is unbiased on exactly the data where the linearised fit is not;
      * weights 1/sqrt(model), because |z|^2 of a tone in complex noise has a
        variance proportional to its own mean - so no point is over-trusted
        merely for being large;
      * a Huber reweighting, so a handful of contaminated points are demoted
        instead of being allowed to drag the whole line;
      * an OPTIONAL second Lorentzian, taken only when it earns its three extra
        degrees of freedom on an F test AND its own amplitude is five standard
        errors from zero AND it sits clear of the main line. This is the honest
        way to handle an interferer that cannot be removed in hardware: MODEL
        it, do not average it into the answer.
      * `exclude`, a list of (centre, half_width) bands dropped before fitting,
        for interferers whose frequency Step 2 has already measured.

    The known-answer test is unchanged and still passes: on a clean line the
    nonlinear fit and the linearised one agree to better than 0.3%. The whole
    difference appears when something else is in the band, which is the case
    this notebook is now in.

    Returns the same keys as before - f0, lw, peak, base, resid, Q - so every
    existing caller keeps working, plus:
      f0_err, lw_err, Q_err   1-sigma from the fit covariance
      n_components            1 or 2
      second                  dict(f0, lw, peak) for the interferer, or None
      second_test             why the second component was or was not taken
      asymmetry               fractional left/right linewidth disagreement
      excluded                points dropped by `exclude`
      model_t, slope          internal parameters, so plots can rebuild the model
    """
    if allow_second is None:
        allow_second = bool(globals().get('FIT_ALLOW_SECOND', True))
    if second_f is None:
        second_f = float(globals().get('FIT_SECOND_F', 20.0))
    if second_t is None:
        second_t = float(globals().get('FIT_SECOND_TSTAT', 5.0))

    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    order = np.argsort(f)
    f, p = f[order], p[order]

    keep = np.ones(len(f), dtype=bool)
    for centre, halfwidth in (exclude or []):
        keep &= np.abs(f - float(centre)) > float(halfwidth)
    n_excluded = int(np.count_nonzero(~keep))
    if int(np.count_nonzero(keep)) < minpts + 3:
        keep = np.ones(len(f), dtype=bool)      # refuse to exclude everything
        n_excluded = 0
    ff, pp = f[keep], p[keep]
    if len(ff) < minpts:
        raise ValueError('fit failed: only %d usable points above the guard'
                         % len(ff))

    t0 = _lorentz_seed(ff, pp)
    try:
        g = _legacy_linear_fit(ff, pp, guard, minpts)
        if np.isfinite(g[1]) and g[1] > 0.0:
            t0 = list(g)
    except Exception:
        pass

    t1, r1, J1, _w1 = _fit_once(ff, pp, t0, 0, allow_slope, robust)
    best = (t1, r1, J1, 0, allow_slope)
    second_stat = None

    if allow_second:
        seed2 = _seed_second_peak(ff, pp, t1, allow_slope)
        if seed2 is not None:
            t2, r2, J2, _w2 = _fit_once(ff, pp, list(t1) + seed2, 1,
                                        allow_slope, robust)
            o = 5 if allow_slope else 4
            sep_ok = abs(t2[o] - t2[0]) > 0.6 * t2[1]
            amp_ok = 0.0 < t2[o + 2] < 0.9 * t2[2]
            wid_ok = 0.1 * t2[1] < t2[o + 1] < 40.0 * t2[1]
            #  Three extra free parameters ALWAYS reduce a residual a little, so
            #  a ratio alone is not evidence. Compare the drop in the sum of
            #  squares with the scatter that is left - the F statistic for
            #  nested models - and require the new amplitude to be many standard
            #  errors from zero as well. Below either threshold the notebook
            #  keeps the simpler model and SAYS SO rather than quietly splitting
            #  a noisy line into two.
            n = len(ff)
            dof = max(n - len(t2), 1)
            s1, s2 = r1 ** 2 * n, r2 ** 2 * n
            fstat = ((s1 - s2) / 3.0) / max(s2 / dof, 1e-300)
            tstat = _t_statistic(J2, r2, n, len(t2), o + 2, t2[o + 2])
            take = (fstat > second_f and tstat > second_t
                    and r2 < second_gain * r1 and sep_ok and amp_ok and wid_ok)
            if take:
                best = (t2, r2, J2, 1, allow_slope)
            second_stat = dict(fstat=float(fstat), amp_tstat=float(tstat),
                               resid_one=float(r1), resid_two=float(r2),
                               separated=bool(sep_ok), amplitude_sane=bool(amp_ok),
                               width_sane=bool(wid_ok), accepted=bool(take),
                               candidate_f0=float(t2[o]),
                               candidate_lw=float(abs(t2[o + 1])),
                               candidate_peak=float(t2[o + 2]))

    t, resid, JtJ, n_extra, slope = best
    err = np.zeros(len(t)) + np.nan
    if JtJ is not None and len(ff) > len(t):
        try:
            cov = np.linalg.inv(JtJ) * (resid ** 2) * len(ff) / (len(ff) - len(t))
            err = np.sqrt(np.maximum(np.diag(cov), 0.0))
        except np.linalg.LinAlgError:
            pass

    f0, lw = float(t[0]), float(abs(t[1]))
    o = 5 if slope else 4
    q_err = float('nan')
    if np.isfinite(err[0]) and np.isfinite(err[1]):
        q_err = abs(f0 / lw) * np.sqrt((err[0] / f0) ** 2 + (err[1] / lw) ** 2)

    out = dict(f0=f0, lw=lw, peak=float(t[2]), base=float(t[3]), resid=resid,
               Q=f0 / lw, f0_err=float(err[0]), lw_err=float(err[1]),
               peak_err=float(err[2]), Q_err=q_err,
               n_components=1 + n_extra, excluded=n_excluded,
               n_points=int(len(ff)), model_t=np.array(t), slope=bool(slope),
               second_test=second_stat,
               second=(dict(f0=float(t[o]), lw=float(abs(t[o + 1])),
                            peak=float(t[o + 2])) if n_extra else None))
    out['asymmetry'] = _flank_asymmetry(ff, pp, out) if diagnostics else float('nan')
    return out


def fit_summary(fit, label='fit', indent='  '):
    """One block of text that says what was fitted and what to distrust."""
    lines = ['%s%-22s : %.6f MHz  +- %.3f kHz'
             % (indent, label + ' centre', fit['f0'] / 1e6,
                fit.get('f0_err', float('nan')) / 1e3),
             '%s%-22s : %.2f kHz  +- %.2f kHz'
             % (indent, 'linewidth', fit['lw'] / 1e3,
                fit.get('lw_err', float('nan')) / 1e3),
             '%s%-22s : %.1f  +- %.1f'
             % (indent, 'quality factor', fit['Q'], fit.get('Q_err', float('nan'))),
             '%s%-22s : %.4f of peak  (%d points, %d excluded)'
             % (indent, 'fit residual', fit['resid'], fit.get('n_points', 0),
                fit.get('excluded', 0))]
    a = fit.get('asymmetry', float('nan'))
    if np.isfinite(a):
        lines.append('%s%-22s : %.1f%%  (left flank against right)'
                     % (indent, 'flank disagreement', 100 * a))
    if fit.get('second'):
        s = fit['second']
        lines.append('%s%-22s : %.6f MHz, %.1f kHz wide, %.1f%% of the peak'
                     % (indent, 'second component', s['f0'] / 1e6, s['lw'] / 1e3,
                        100 * s['peak'] / max(fit['peak'], 1e-300)))
        lines.append('%s%-22s : the numbers above are the MAIN line with that'
                     % (indent, ''))
        lines.append('%s%-22s : component fitted out, not averaged in.'
                     % (indent, ''))
    elif fit.get('second_test') is not None:
        st = fit['second_test']
        lines.append('%s%-22s : declined (F = %.1f, amplitude %.1f sigma)'
                     % (indent, 'second component', st['fstat'], st['amp_tstat']))
    return '\n'.join(lines)


def lorentzian(f, fit):
    """The fitted model evaluated anywhere - INCLUDING any second component.

    Used by every plot, so no cell has to rebuild it and none can rebuild it
    differently. This is what should be drawn through the measured points."""
    f = np.asarray(f, dtype=float)
    t = fit.get('model_t')
    if t is None:
        return fit['peak'] / (1.0 + (2.0 * (f - fit['f0']) / fit['lw']) ** 2) \
               + fit['base']
    return _lorentz_model(f, t, fit.get('n_components', 1) - 1,
                          fit.get('slope', False))


def lorentzian_main(f, fit):
    """The COIL alone: the interferer and the floor removed.

    This is the curve that belongs in anything downstream that assumes ONE
    resonance - the linewidth comparison in Step 5, the feedback model in Step
    8, the numbers written to JSON. Drawing this against the measured points is
    also the clearest picture of how much the interferer was worth."""
    f = np.asarray(f, dtype=float)
    return fit['peak'] / (1.0 + (2.0 * (f - fit['f0']) / fit['lw']) ** 2)


def lorentzian_second(f, fit):
    """The interferer alone, or zeros if the fit did not take one."""
    f = np.asarray(f, dtype=float)
    s = fit.get('second')
    if not s:
        return np.zeros(len(f))
    return s['peak'] / (1.0 + (2.0 * (f - s['f0']) / s['lw']) ** 2)


def receiver_policy(lw, weak=False, high_q=False, requested=None):
    """Choose a zero-IF receiver for noise/runtime, not apparent linewidth.

    Generator and LO use the same word at every point, so the wanted tone is
    always at DC: CIC/FIR passband width does not convolve the swept resonance.
    Higher decimation buys coherent SNR but costs FIR settling time."""
    if requested is not None:
        dec = int(requested)
    elif weak or high_q or (np.isfinite(lw) and lw < 0.5*globals().get('LW', lw)):
        dec = 12
    else:
        dec = 10
    return int(min(12, max(8, dec)))


def choose_dec(lw):
    """Compatibility wrapper; new code should call receiver_policy()."""
    return receiver_policy(lw, high_q=(np.isfinite(lw) and lw < 0.5*globals().get('LW', lw)))


def adaptive_settle(lw, dec_log2, multiplier=1.0):
    """Conservative zero-IF filter plus resonator settling time."""
    fout = FS_CLK / float(1 << int(dec_log2))
    fir = FIR_SETTLE_GROUP_DELAYS * 0.5*(FIR_TAPS-1) / fout
    resonator = HIGH_Q_RESONATOR_TAU / (np.pi*max(float(lw), 1.0))
    return float(multiplier * max(SETTLE, fir, resonator))


def adaptive_frequency_grid(f0, lw, points_per_lw=None, wing_lw=None):
    """Dense nonuniform centre plus sparse wings for a stable background fit."""
    ppl = int(HIGH_Q_POINTS_PER_LW if points_per_lw is None else points_per_lw)
    wing = float(HIGH_Q_WING_LW if wing_lw is None else wing_lw)
    lw = max(float(lw), 1.0)
    centre = np.linspace(-1.25, 1.25, int(np.ceil(2.5*ppl))+1)
    left = np.linspace(-wing, -1.25, max(10, ppl//2), endpoint=False)
    right = -left[::-1]
    return float(f0) + np.unique(np.r_[left, centre, right])*lw


def receiver_metadata(dec_log2, lw, settle_s, f=None, fit=None):
    status = int(acq.status2())
    points = 0
    if f is not None and fit is not None:
        points = int(np.sum(np.abs(np.asarray(f)-fit['f0']) <= 0.5*fit['lw']))
    return {'dec_log2': int(dec_log2), 'decimation': int(1 << int(dec_log2)),
            'output_rate_hz': float(FS_CLK/(1 << int(dec_log2))),
            'fir_bypassed': bool(status & 0x4), 'fir_overrun': bool(status & 0x2),
            'dma_overflow': bool(status & 0x1), 'write_error': bool(status & 0x8),
            'status2': status, 'settle_s': float(settle_s),
            'points_per_linewidth': points, 'zero_if_tracked': True}


def half_power_width(f, p, base=None):
    """Interpolated FWHM using a supplied fit floor or the sweep edges."""
    f = np.asarray(f, dtype=float); p = np.asarray(p, dtype=float)
    order = np.argsort(f); f, p = f[order], p[order]
    edge_n = max(2, len(p)//10)
    if base is None:
        base = float(np.median(np.r_[p[:edge_n], p[-edge_n:]]))
    peak_i = int(np.argmax(p)); half = float(base) + 0.5*(p[peak_i]-float(base))
    def _cross(indices):
        for i, j in indices:
            if (p[i]-half)*(p[j]-half) <= 0 and p[j] != p[i]:
                return float(f[i] + (half-p[i])*(f[j]-f[i])/(p[j]-p[i]))
        return None
    left = _cross((i, i+1) for i in range(peak_i-1, -1, -1))
    right = _cross((i, i+1) for i in range(peak_i, len(f)-1))
    return None if left is None or right is None else float(right-left)


def fit_complex_resonator(f, z, seed):
    """Complex pole plus linear complex background, using NumPy only."""
    f = np.asarray(f, dtype=float); z = np.asarray(z, dtype=complex)
    f0, lw = float(seed['f0']), float(seed['lw'])
    span = max(float(np.ptp(f)), lw); best = None
    for frac in (0.30, 0.10, 0.03):
        fgrid = np.linspace(f0-frac*lw, f0+frac*lw, 13)
        lwgrid = np.linspace(max(0.2*lw, (1-frac)*lw), (1+frac)*lw, 13)
        for fc in fgrid:
            x = (f-fc)/span
            for width in lwgrid:
                for sign in (-1.0, 1.0):
                    pole = 1.0/(1.0 + sign*2.0j*(f-fc)/max(width, 1e-30))
                    design = np.column_stack([np.ones(len(f)), x, pole])
                    coef, _, _, _ = np.linalg.lstsq(design, z, rcond=-1)
                    model = np.dot(design, coef)
                    rms = float(np.sqrt(np.mean(np.abs(z-model)**2)) /
                                max(np.max(np.abs(z)), 1e-30))
                    if best is None or rms < best[0]:
                        best = (rms, float(fc), float(width), coef, model, sign)
        _, f0, lw, _, _, _ = best
    return {'f0': f0, 'lw': lw, 'Q': f0/lw, 'resid': best[0],
            'background_constant': best[3][0],
            'background_slope': best[3][1], 'resonance_amplitude': best[3][2],
            'model': best[4], 'pole_sign': best[5]}


def trace_fit_diagnostics(f, p, fit, z=None, groups=5):
    """Independent width, residual vector and delete-group fit uncertainty."""
    f = np.asarray(f, dtype=float); p = np.asarray(p, dtype=float)
    estimates = []
    for group in range(groups):
        keep = (np.arange(len(f)) % groups) != group
        try:
            estimates.append(fit_lorentzian(f[keep], p[keep]))
        except Exception:
            pass
    def _unc(key):
        vals = np.asarray([x[key] for x in estimates], dtype=float)
        return float(np.std(vals, ddof=1)) if len(vals) >= 2 else float('nan')
    scale = max(float(np.max(p)), 1e-30)
    hp = half_power_width(f, p, base=fit.get('base'))
    #  OFF BY DEFAULT, AND IT COSTS NOTHING TO TURN IT OFF.  This grid search is
    #  three passes of 13 x 13 x 2 least-squares solves - about a thousand per
    #  trace - and _verify_point() takes six traces per point. On the board that
    #  is most of the campaign's runtime, and the last run had to be interrupted
    #  by hand inside this very function. What it computes is a complex-pole
    #  width, which needs the phase to mean something from point to point, and
    #  Step 5 explains at length that on this board it does not: sweep() retunes
    #  two oscillators with two separate register writes and nothing preserves
    #  the phase between them. A thousand solves per trace, for a number that
    #  _fit_quality() already refuses to gate on.
    cfit = (fit_complex_resonator(f, z, fit)
            if (z is not None and bool(globals().get('COMPLEX_FIT_DIAGNOSTIC', False)))
            else None)
    residual = (p - lorentzian(f, fit)) / scale
    kernel = np.ones(5, dtype=float) / 5.0
    smooth = np.convolve(residual, kernel, mode='valid') if len(residual) >= 5 else residual
    return {'half_power_lw': hp,
            'fitted_peak': float(fit.get('peak', float('nan'))),
            'n_components': int(fit.get('n_components', 1)),
            'flank_asymmetry': float(fit.get('asymmetry', float('nan'))),
            'lw_err': float(fit.get('lw_err', float('nan'))),
            'f0_err': float(fit.get('f0_err', float('nan'))),
            'q_err': float(fit.get('Q_err', float('nan'))),
            'second_component': (dict(fit['second']) if fit.get('second') else None),
            'half_power_disagreement': (abs(hp/fit['lw']-1.0)
                                        if hp is not None else float('inf')),
            'residual': residual,
            'residual_rms': float(np.sqrt(np.mean(residual**2))),
            'residual_max': float(np.max(np.abs(residual))),
            'structured_residual': float(np.sqrt(np.mean(smooth**2))),
            'floor_peak_amp': float(np.sqrt(max(fit['base'], 0.0))/
                                    max(np.sqrt(max(fit['peak'], 0.0)), 1e-30)),
            'complex_fit': cfit,
            #  nan, NOT inf. inf reads as "disagrees infinitely" to every
            #  comparison downstream, so switching the complex fit off would
            #  otherwise look like a catastrophic failure rather than a
            #  measurement that was not taken.
            'complex_lw_disagreement': (abs(cfit['lw']/fit['lw']-1.0)
                                        if cfit is not None else float('nan')),
            'f0_uncertainty': _unc('f0'), 'lw_uncertainty': _unc('lw'),
            'q_uncertainty': _unc('Q'), 'jackknife_fits': len(estimates)}


def background_sweep(freqs, dec_log2, navg=None, settle=None, verbose=False):
    """The same grid, the same receiver, the SAME tuning words - drive OFF.

    Anything that arrives without being driven - a board clock product, room
    pickup, amplifier noise shaped by the standing waves on the several-metre
    wiring - lands in |z|^2 whether or not the coil is being driven. It adds in
    POWER rather than coherently, because sweep() rewrites both oscillators
    between points and the phase between them is not preserved (Step 5). So it
    can be measured on exactly the grid the fit will use, and subtracted there.

    This is BOTH the correction and the diagnosis, which is why it is worth the
    extra sweep:

        subtract it and a DRIVE-INDEPENDENT interferer disappears completely;
        subtract it and a DRIVEN second mode stays exactly where it was.

    Step 3 prints which of those happened, so the next thing to do at the bench
    is decided by a measurement rather than by argument."""
    navg = N_AVG if navg is None else navg
    settle = SETTLE if settle is None else settle
    gen.off()
    acq.start(int(dec_log2))
    bg = np.zeros(len(freqs))
    t0 = time.time()
    try:
        for k, f in enumerate(freqs):
            gen.freq(f)                       # sets the WORD; amplitude stays 0
            acq.tune_word(gen.lo_word())      # identical tuning to the driven sweep
            time.sleep(settle)
            bg[k] = float(abs(acq.measure(navg)) ** 2)
    finally:
        gen.off()
    if verbose:
        print('  undriven background: %d points in %.0f s, median %.3e, '
              'max %.3e' % (len(freqs), time.time() - t0,
                            float(np.median(bg)), float(np.max(bg))))
    return bg


def refine_and_fit(f_centre, lw_hint, drive, npts=61, span_lw=2.5, verbose=True,
                   subtract_background=None, exclude=None, allow_second=None):
    """Dense sweep at a decimation chosen from lw_hint, then fit.

    Returns (dec_log2, f, p, fit). Called by Step 3 and AGAIN by Step 4b once the
    drive has been adopted, so the headline numbers always belong to the drive the
    rest of the notebook uses. Nothing about it is specific to this coil, which is
    the point: the same call works warm, cold, in the trap or out of it.

    Three things are new, all of them because of what the 4 K long-wire cooldown
    put in the band next to the resonance:

      `subtract_background` measures the SAME grid with the drive off and takes
      it off in power before fitting. Defaults to SUBTRACT_BACKGROUND.
      `exclude` drops bands around interferers Step 2 has already located.
      `allow_second` lets the fit carry an interferer it cannot drop.

    The returned `p` is what was FITTED (background already removed); the raw
    driven trace and the background are attached to the fit as `p_raw` and
    `background`, so a plot can always show all three."""
    if subtract_background is None:
        subtract_background = bool(globals().get('SUBTRACT_BACKGROUND', False))
    if exclude is None:
        exclude = list(globals().get('FIT_EXCLUDE_BANDS', []) or [])

    dec = receiver_policy(lw_hint, high_q=(lw_hint < 0.5*globals().get('LW', lw_hint)))
    acq.start(dec)
    if verbose:
        print('  zero-IF receiver R = %d, output rate %.1f kHz, FIR corrected; '
              'tracked tone remains at DC' % (1 << dec, acq.f_out() / 1e3))
    grid = adaptive_frequency_grid(f_centre, lw_hint,
                                   points_per_lw=max(HIGH_Q_POINTS_PER_LW, npts//3),
                                   wing_lw=max(HIGH_Q_WING_LW, span_lw))
    use_settle = adaptive_settle(lw_hint, dec)
    if bool(globals().get('SWEEP_SHUFFLE', False)):
        #  Visit the grid in random order, then sort. Anything that drifts
        #  DURING the sweep is spread across the whole line instead of piling up
        #  on the end that happened to be measured last, so it becomes scatter
        #  in Q rather than bias in Q. Step 3b is the measurement that says
        #  whether there is any drift to spread.
        _perm = np.random.RandomState(1234567).permutation(len(grid))
        f, z = sweep(grid[_perm], drive_frac=drive, navg=N_AVG,
                     settle=use_settle, level_guard=0.95)
    else:
        f, z = sweep(grid, drive_frac=drive, navg=N_AVG, settle=use_settle,
                     level_guard=0.95)
    gen.amp(0.0)
    _o = np.argsort(f)
    f = f[_o]
    z = z[_o]
    p_raw = np.abs(z) ** 2

    bg = None
    p = p_raw
    if subtract_background:
        bg = background_sweep(f, dec, navg=N_AVG, settle=use_settle, verbose=verbose)
        #  Never let the subtraction manufacture negative power: the background
        #  is a MEASUREMENT with its own scatter, so clip at a small positive
        #  floor rather than at zero.
        p = np.maximum(p_raw - bg, 1e-6 * float(np.max(p_raw)))
        acq.start(dec)

    fit = fit_lorentzian(f, p, exclude=exclude, allow_second=allow_second)
    fit['p_raw'] = p_raw
    fit['background'] = bg
    fit['background_subtracted'] = bool(subtract_background)
    if bg is not None:
        #  How much of the contamination was drive-independent? A number, not
        #  an opinion: refit the RAW trace and compare the widths.
        try:
            raw_fit = fit_lorentzian(f, p_raw, exclude=exclude,
                                     allow_second=allow_second, diagnostics=False)
            fit['lw_without_subtraction'] = float(raw_fit['lw'])
            fit['background_lw_change'] = float(raw_fit['lw'] / fit['lw'] - 1.0)
        except Exception:
            pass
    return dec, f, p, fit


def emulator_high_q_qualification():
    """Exercise zero-IF tracking, FIR/CIC status, density and settling over Q/f0."""
    rows = []
    try:
        ps.off()
        for requested_f0 in HIGH_Q_TEST_CENTRES_HZ:
            for requested_q in HIGH_Q_TEST_VALUES:
                f0q, qq = emu.config(requested_f0, requested_q, delay=12, oshift=7)
                lwq = f0q/qq; emu.clear(); emu.enable()
                for dec in HIGH_Q_TEST_DECIMATIONS:
                    acq.start(dec); settle = adaptive_settle(lwq, dec)
                    grid = adaptive_frequency_grid(f0q, lwq)
                    f,z = sweep(grid, drive_frac=SELFTEST_DRIVE, navg=N_AVG,
                                settle=settle, level_guard=0.95)
                    fit = fit_lorentzian(f,np.abs(z)**2)
                    meta = receiver_metadata(dec,fit['lw'],settle,f=f,fit=fit)
                    row={'f0_known':float(f0q),'q_known':float(qq),'dec_log2':int(dec),
                         'f0_meas':float(fit['f0']),'q_meas':float(fit['Q']),
                         'centre_error_lw':float((fit['f0']-f0q)/lwq),
                         'q_error_frac':float(fit['Q']/qq-1.0),'receiver':meta}
                    if dec == 12:
                        f2,z2=sweep(grid,drive_frac=SELFTEST_DRIVE,navg=N_AVG,
                                    settle=2.0*settle,level_guard=0.95)
                        fit2=fit_lorentzian(f2,np.abs(z2)**2)
                        row['double_settle_q_change']=float(abs(fit2['Q']/fit['Q']-1.0))
                        row['double_settle_f0_change_lw']=float(abs(fit2['f0']-fit['f0'])/lwq)
                    row['ok']=bool(meta['points_per_linewidth']>=HIGH_Q_POINTS_PER_LW and
                        not meta['fir_bypassed'] and not meta['fir_overrun'] and
                        abs(row['centre_error_lw'])<=0.02 and abs(row['q_error_frac'])<=0.03 and
                        row.get('double_settle_q_change',0)<=SETTLING_CONVERGENCE_TOL and
                        row.get('double_settle_f0_change_lw',0)<=SETTLING_CONVERGENCE_TOL)
                    rows.append(row)
    finally:
        emu.disable(); gen.off(); ps.off()
    # Cross-decimation agreement is evaluated per emulated resonance.
    for f0q in sorted({round(r['f0_known'],3) for r in rows}):
        group=[r for r in rows if round(r['f0_known'],3)==f0q]
        for qq in sorted({round(r['q_known'],3) for r in group}):
            same=[r for r in group if round(r['q_known'],3)==qq]
            spread=np.ptp([r['q_meas'] for r in same])/max(np.mean([r['q_meas'] for r in same]),1e-30)
            for r in same: r['decimation_spread_frac']=float(spread); r['ok']=bool(r['ok'] and spread<=0.03)
    return rows


def measure_transfer(f0, drive):
    """One-way DAC -> coil -> ADC transfer at f0.

    Returns (raw_peak, tone, G_fwd). raw_peak is the TOTAL excursion, inflated by
    the converter's DC offset and everything else in the record - that is the
    right clipping guard and the WRONG way to size a tone. tone is the component
    at f0 alone, and only that is a transfer."""
    try:
        gen.freq(f0); gen.amp(drive); gen.run(); time.sleep(0.05)
        raw = sc.peak_frac(SC_NMAX)
        tone = sc.tone_frac(f0)
    finally:
        gen.off()
    return raw, tone, tone / max(drive, 1e-12)


print('analysis helpers defined')
print('  lineshape fit: nonlinear least squares, robust weights, optional second')
print('  component (F > %.0f and amplitude > %.0f sigma); background subtraction %s'
      % (globals().get('FIT_SECOND_F', 20.0), globals().get('FIT_SECOND_TSTAT', 5.0),
         'ON' if globals().get('SUBTRACT_BACKGROUND', False) else 'off'))

def amp_out_dbm(total_dac_fraction):
    return (10.0 + 20.0*np.log10(max(float(total_dac_fraction),1e-12))
            - EXT_PAD_DB + EXT_GAIN_SAFETY_DB)

def amp_dac_fraction(dbm):
    return float(10.0**((float(dbm)-10.0+EXT_PAD_DB-EXT_GAIN_SAFETY_DB)/20.0))

def _background_on(grid):
    bg=FIT.get('background')
    if bg is None:
        return np.zeros(len(grid))
    return np.interp(grid,f3,np.asarray(bg),left=bg[0],right=bg[-1])

def measured_fit(gain, phase, drive, centre, linewidth, npts=61):
    grid=adaptive_frequency_grid(centre,linewidth,points_per_lw=25,wing_lw=4)
    dec=receiver_policy(linewidth)
    acq.start(dec)
    ps.config_delay(F0); ps.set(gain,phase)
    try:
        f,z=sweep(grid,drive_frac=drive,navg=FB_NAVG,
                  settle=adaptive_settle(linewidth,dec),
                  level_guard=FB_LEVEL_GUARD)
    finally:
        gen.off(); ps.off(); acq.stop()
    p=np.maximum(np.abs(z)**2-_background_on(f),1e-12)
    ft=fit_lorentzian(f,p,allow_second=True)
    if ft['resid']>SEARCH_FIT_MAX_RESID or not (0.02*LW<ft['lw']<50*LW):
        raise RuntimeError('feedback fit rejected: width %.1f kHz residual %.3f' %
                           (ft['lw']/1e3,ft['resid']))
    return f,p,ft


---
# Step 0 â€” preflight and known answer

> ### âœ… PUMP ON
> Estimated bench time: **about 1 minute**. The emulator replaces the digitised input; the physical coil may remain connected.


In [ ]:
pump('on', 'STEP 0 - preflight and known answer', 1.0)
all_off()
if acq.build_id() != EXPECTED_BUILD_ID:
    raise RuntimeError('wrong FPGA build: expected 0x%08X, got 0x%08X' %
                       (EXPECTED_BUILD_ID, acq.build_id()))
gen.assert_b_zero()
ps.monitor('input', shift=0)
if (ps.readback()['monitor'] & 0x3f) != 1:
    raise RuntimeError('OUT2 input monitor did not read back')
ps.monitor_off()
f_known, q_known = emu.config(SELFTEST_F0, SELFTEST_Q, delay=12, oshift=7)
try:
    emu.clear(); emu.enable(); ps.off(); acq.start(DEC_FIT)
    lw_known = f_known/q_known
    grid = adaptive_frequency_grid(f_known, lw_known, points_per_lw=20, wing_lw=4)
    f_t, z_t = sweep(grid, drive_frac=SELFTEST_DRIVE, navg=N_AVG,
                     settle=adaptive_settle(lw_known, DEC_FIT), level_guard=0.95)
    fit_t = fit_lorentzian(f_t, np.abs(z_t)**2, allow_second=False)
finally:
    all_off()
_qerr = abs(fit_t['Q']/q_known-1.0)
_ferr = abs(fit_t['f0']-f_known)/lw_known
if _qerr > 0.03 or _ferr > 0.02 or fit_t['resid'] > SEARCH_FIT_MAX_RESID:
    raise RuntimeError('known-answer failed: Q error %.1f%%, centre %.3f lw, residual %.3f' %
                       (100*_qerr, _ferr, fit_t['resid']))
print('PASS: known Q %.1f, measured %.1f; hardware safe' % (q_known, fit_t['Q']))


---
# Step 1 â€” input safety

> ### âœ… PUMP ON
> Estimated bench time: **under 30 seconds**. The real IN1 is checked with drive off and at the starting drive.


In [ ]:
pump('on', 'STEP 1 - input safety', 0.3)
all_off()
quiet_frac = sc.peak_frac(GUARD_NSAMP)
try:
    gen.freq(F0_SEED_HZ); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.05)
    driven_frac = sc.peak_frac(GUARD_NSAMP)
finally:
    all_off()
print('IN1 drive off %.4f full scale; driven %.4f full scale' % (quiet_frac, driven_frac))
if quiet_frac > 0.50 or driven_frac > FB_LEVEL_GUARD:
    raise RuntimeError('unsafe IN1 level; investigate before continuing')
print('PASS: input levels safe; hardware safe')


---
# Step 2 â€” fresh resonance and Q

> ### âœ… PUMP ON
> Estimated bench time: **about 2 minutes**. A bounded local search is confirmed twice, bracketed, background-subtracted and robustly fitted.


In [ ]:
pump('on', 'STEP 2 - fresh resonance and Q', 2.0)
all_off()
seed_lw = F0_SEED_HZ/Q_SEED
acq.start(DEC_SEARCH)
FOUND = None
for span_lw, npts in ((3.0, 41), (8.0, 61), (20.0, 81)):
    grid = F0_SEED_HZ + np.linspace(-span_lw, span_lw, npts)*seed_lw
    reps = []
    for repeat in range(2):
        f_r, z_r = sweep(grid, drive_frac=DRIVE_FRAC, navg=32,
                         settle=0.008, level_guard=FB_LEVEL_GUARD)
        reps.append(np.abs(z_r)**2)
    a = reps[0]-np.mean(reps[0]); b = reps[1]-np.mean(reps[1])
    denom = np.sqrt(np.dot(a,a)*np.dot(b,b))
    corr = float(np.dot(a,b)/denom) if denom else 0.0
    p_r = 0.5*(reps[0]+reps[1])
    rough_lw = half_power_width(grid, p_r)
    print('  +/-%g seed linewidths: repeat correlation %.3f, rough width %s' %
          (span_lw, corr, 'unresolved' if rough_lw is None else '%.1f kHz'%(rough_lw/1e3)))
    if corr >= SEARCH_MIN_CORR and rough_lw:
        FOUND = {'f0': float(grid[np.argmax(p_r)]), 'lw': float(rough_lw),
                 'corr': corr}
        break
all_off()
if FOUND is None:
    raise RuntimeError('line not confirmed in bounded local search; correct F0_SEED_HZ/Q_SEED')
DEC_USE, f3, p3, FIT = refine_and_fit(FOUND['f0'], FOUND['lw'], DRIVE_FRAC,
                                      npts=81, span_lw=5.0, allow_second=True)
F0, LW, Q_SWEEP = FIT['f0'], FIT['lw'], FIT['Q']
diag = trace_fit_diagnostics(f3, p3, FIT)
hp = diag['half_power_lw']
edge_ok = (p3[0] < FIT['base']+0.5*FIT['peak'] and
           p3[-1] < FIT['base']+0.5*FIT['peak'])
hp_err = abs(hp/LW-1.0) if hp else float('inf')
points_in_lw = int(np.sum(np.abs(f3-F0) <= 0.5*LW))
if (FIT['resid'] > SEARCH_FIT_MAX_RESID or not edge_ok or hp_err > 0.15 or
        points_in_lw < 10):
    all_off()
    raise RuntimeError('Q rejected: residual %.3f, flanks %s, half-power error %.1f%%, %d points/lw' %
                       (FIT['resid'], edge_ok, 100*hp_err, points_in_lw))
print('MEASURED f0 %.6f MHz, linewidth %.2f kHz, Q %.1f' % (F0/1e6, LW/1e3, Q_SWEEP))
print('repeat corr %.3f, residual %.4f, half-power agreement %.1f%%, %d points/lw' %
      (FOUND['corr'], FIT['resid'], 100*hp_err, points_in_lw))
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(8,6))
fine=np.linspace(f3.min(),f3.max(),600)
ax1.plot((f3-F0)/1e3,p3,'o',ms=3,label='measured')
ax1.plot((fine-F0)/1e3,lorentzian(fine,FIT),'-',label='robust fit')
ax1.set_ylabel('power'); ax1.legend(); ax1.set_title('Fresh open-loop resonance')
res=(p3-lorentzian(f3,FIT))/max(np.max(p3),1e-30)
ax2.plot((f3-F0)/1e3,res,'o-'); ax2.axhline(0,color='0.5')
ax2.set_xlabel('offset from fitted centre / kHz'); ax2.set_ylabel('residual / peak')
plt.show()
all_off()


---
# Step 3 â€” short drive window

> ### âœ… PUMP ON
> Estimated bench time: **about 2 minutes**. Only three drive rungs are measured; the highest clean linear rung is adopted.


In [ ]:
pump('on', 'STEP 3 - short drive window', 2.0)
rows=[]
for fraction in (0.50,0.71,1.00):
    drive=DRIVE_FRAC*fraction
    try:
        dec,f,p,ft=refine_and_fit(F0,LW,drive,npts=61,span_lw=4.0,
                                  allow_second=True)
        try:
            gen.freq(ft['f0']); gen.amp(drive); gen.run(); time.sleep(0.03)
            adc=sc.peak_frac(GUARD_NSAMP)
        finally:
            all_off()
        good=(ft['resid']<=SEARCH_FIT_MAX_RESID and adc<FB_LEVEL_GUARD and
              abs(ft['f0']-F0)<0.25*LW and abs(ft['lw']/LW-1.0)<0.15)
        rows.append({'fraction':fraction,'drive':drive,'fit':ft,'f':f,'p':p,
                     'adc':adc,'good':good})
        print('  %.2f: drive %.6f, Q %.1f, residual %.3f, ADC %.3f, %s' %
              (fraction,drive,ft['Q'],ft['resid'],adc,'PASS' if good else 'reject'))
    except Exception as exc:
        all_off(); print('  %.2f rejected: %s'%(fraction,exc))
good_rows=[r for r in rows if r['good']]
if not good_rows:
    raise RuntimeError('no trustworthy drive rung; feedback calibration locked')
chosen=max(good_rows,key=lambda r:r['drive'])
DRIVE_FRAC=chosen['drive']; FIT=chosen['fit']; f3=chosen['f']; p3=chosen['p']
F0,LW,Q_SWEEP=FIT['f0'],FIT['lw'],FIT['Q']
print('ADOPTED drive %.6f; fresh f0 %.6f MHz, Q %.1f' % (DRIVE_FRAC,F0/1e6,Q_SWEEP))
all_off()


---
# Step 4 â€” physical OUT2 verification

> ### âœ… PUMP ON
> Estimated bench time: **about 1 minute plus analyser setup**. Connect OUT2 to the analyser. The full resonance grid is replayed under max hold.


In [ ]:
pump('on', 'STEP 4 - physical IN1 to OUT2 replay', 1.0)
print('Set analyser: centre %.6f MHz; span %.6f..%.6f MHz; RBW <= %.2f kHz;' %
      (F0/1e6,f3.min()/1e6,f3.max()/1e6,LW/1e4))
print('Use logarithmic power and MAX HOLD. Clear max hold before replay.')
input('Hardware is OFF. Press Enter when analyser is ready: ')
all_off()
clipped=False
try:
    gen.freq(F0); gen.amp(DRIVE_FRAC); gen.run(); gen.assert_b_zero()
    ps.config_delay(F0); ps.off(); ps.monitor('input',shift=OUT2_MON_SHIFT)
    expected=ps.readback()
    for rep in range(OUT2_REPEATS):
        for frequency in f3:
            gen.freq(float(frequency)); time.sleep(OUT2_DWELL)
            if sc.peak_frac(GUARD_NSAMP)>FB_LEVEL_GUARD:
                raise RuntimeError('IN1 level guard tripped')
            if acq.build_id()!=EXPECTED_BUILD_ID: raise RuntimeError('build ID changed')
            ps.assert_held(expected); gen.assert_b_zero()
        print('  replay %d/%d'%(rep+1,OUT2_REPEATS))
    clipped=ps.monitor_clipped()
finally:
    try: clipped=clipped or ps.monitor_clipped()
    finally: all_off()
if clipped: raise RuntimeError('OUT2 monitor clipped; trace invalid')
answer=input('Is the FULL resonance trace visible from OUT2? Type YES: ').strip()
if answer!='YES': raise RuntimeError('analogue OUT2 path not confirmed; feedback locked')
OUT2_ANALOGUE_CONFIRMED=True
print('PASS: physical OUT2 replay confirmed; hardware safe')


---
# Step 5 â€” local narrowing-phase refinement

> ### ðŸ›‘ PUMP OFF
> Estimated bench time: **about 3 minutes**. Stop the pump and let vibration settle. Five local phases and their opposites are measured.


In [ ]:
pump('off', 'STEP 5 - local narrowing-phase refinement', 3.0)
if not OUT2_ANALOGUE_CONFIRMED: raise RuntimeError('Step 4 OUT2 confirmation required')
phase_rows=[]
for offset in PHASE_LOCAL_OFFSETS:
    phi=(PHASE_SEED_DEG+offset)%360.0
    fp=measured_fit(PHASE_PROBE_GAIN,phi,DRIVE_FRAC,F0,LW)[2]
    fm=measured_fit(PHASE_PROBE_GAIN,phi+180.0,DRIVE_FRAC,F0,LW)[2]
    kval=(fp['Q']-fm['Q'])/(fp['Q']+fm['Q'])
    phase_rows.append({'offset':offset,'phase':phi,'K':kval,'Qp':fp['Q'],'Qm':fm['Q']})
    print('  phase %6.1f / %6.1f: Q+ %.1f Q- %.1f K %.4f' %
          (phi,(phi+180)%360,fp['Q'],fm['Q'],kval))
best=max(phase_rows,key=lambda r:r['K'])
noise=1.4826*np.median(np.abs(np.asarray([r['K'] for r in phase_rows])-
                              np.median([r['K'] for r in phase_rows])))
if best['offset'] in (min(PHASE_LOCAL_OFFSETS),max(PHASE_LOCAL_OFFSETS)):
    raise RuntimeError('best phase is at local-search boundary; improve PHASE_SEED_DEG')
if best['K']<=max(0.01,3.0*noise):
    raise RuntimeError('feedback phase effect is not significant; improve seed/check loop')
PHI_NARROW=best['phase']; PHI_BROAD=(PHI_NARROW+180.0)%360.0
print('ADOPTED narrowing phase %.1f deg; paired K %.4f; hardware safe' %
      (PHI_NARROW,best['K']))
all_off()


---
# Step 6 â€” short K/A calibration

> ### ðŸ›‘ PUMP OFF
> Estimated bench time: **about 2 minutes**. Two narrowing/broadening gain pairs establish the session's K/A conversion.


In [ ]:
pump('off', 'STEP 6 - short K/A calibration', 2.0)
gains_ok=[]; ks=[]
for gain in CAL_GAINS:
    fp=measured_fit(gain,PHI_NARROW,DRIVE_FRAC,F0,LW)[2]
    fm=measured_fit(gain,PHI_BROAD,DRIVE_FRAC,F0,LW)[2]
    kval=(fp['Q']-fm['Q'])/(fp['Q']+fm['Q'])
    gains_ok.append(float(gain)); ks.append(float(kval))
    print('  A %.2f: Q+ %.1f Q- %.1f -> K %.4f, K/A %.4f' %
          (gain,fp['Q'],fm['Q'],kval,kval/gain))
ka=np.asarray(ks)/np.asarray(gains_ok)
ka_disagreement=np.ptp(ka)/max(abs(np.mean(ka)),1e-12)
if np.any(ka<=0) or ka_disagreement>K_PER_A_LINEARITY_TOL:
    raise RuntimeError('K/A disagreement %.1f%% exceeds %.1f%%' %
                       (100*ka_disagreement,100*K_PER_A_LINEARITY_TOL))
K_PER_A=float(np.sum(np.asarray(gains_ok)*np.asarray(ks))/
              np.sum(np.asarray(gains_ok)**2))
K_REACHABLE=min(MANUAL_K_MAX,FB_GAIN_MAX*K_PER_A)
if K_REACHABLE<=max(ks):
    raise RuntimeError('calibrated manual range does not extend above calibration points')
print('K = %.5f x A; reachable manual K <= %.4f; inferred onset A %.3f' %
      (K_PER_A,K_REACHABLE,1.0/K_PER_A))
all_off()


---
# Step 7 â€” manual K analyser replay

> ### ðŸ›‘ PUMP OFF
> Estimated bench time: **operator controlled; budget 6 minutes**. Each command replays a full predicted resonance grid to OUT2 and captures an internal fit.


In [ ]:
pump('off', 'STEP 7 - manual K analyser replay', 6.0)
MANUAL_LOG=[]
current=None

def manual_setting(target_k):
    k=float(target_k)
    if not (0.0<=k<=K_REACHABLE): raise ValueError('K outside [0, %.4f]'%K_REACHABLE)
    gain=k/K_PER_A
    drive=amp_dac_fraction(AMP_OUT_MAX_DBM)*(1.0-k)
    if gain>FB_GAIN_MAX: raise ValueError('feedback coefficient ceiling')
    if drive*8192<K_LADDER_MIN_COUNTS: raise ValueError('fewer than four DAC counts')
    total=drive/max(1.0-k,1e-9)
    if amp_out_dbm(total)>AMP_P1DB_DBM: raise RuntimeError('planned amplifier level past P1dB')
    return {'target_k':k,'gain':gain,'drive':drive,'amp_dbm':amp_out_dbm(total)}

def replay_k(target_k):
    row=manual_setting(target_k)
    predicted_lw=max(LW*(1.0-row['target_k']),0.02*LW)
    grid=adaptive_frequency_grid(F0,predicted_lw,points_per_lw=25,wing_lw=5)
    print('K %.4f: A %.4f, drive %.6f (%.1f counts), amp %+.1f dBm' %
          (row['target_k'],row['gain'],row['drive'],row['drive']*8192,row['amp_dbm']))
    print('Analyser centre %.6f MHz; span %.6f..%.6f MHz; RBW <= %.3f kHz' %
          (F0/1e6,grid.min()/1e6,grid.max()/1e6,predicted_lw/1e4))
    all_off(); input('Clear analyser MAX HOLD, then press Enter to replay: ')
    z=[]; clipped=False
    try:
        replay_dec=receiver_policy(predicted_lw); acq.start(replay_dec)
        gen.freq(F0); gen.amp(row['drive']); gen.run(); gen.assert_b_zero()
        ps.config_delay(F0); ps.monitor('input',shift=OUT2_MON_SHIFT)
        ci,cq=ps.set(row['gain'],PHI_NARROW); expected=ps.readback()
        for rep in range(OUT2_REPEATS):
            zr=[]
            for frequency in grid:
                gen.freq(float(frequency)); acq.tune_word(gen.lo_word())
                time.sleep(max(OUT2_DWELL,adaptive_settle(predicted_lw,replay_dec)))
                peak=sc.peak_frac(LEVEL_GUARD_NSAMP)
                if peak>FB_LEVEL_GUARD: raise RuntimeError('ADC guard %.3f'%peak)
                ps.assert_held(expected); gen.assert_b_zero()
                if rep==0: zr.append(acq.measure(FB_NAVG))
            if rep==0: z=zr
            print('  replay %d/%d'%(rep+1,OUT2_REPEATS))
        clipped=ps.monitor_clipped()
    except Exception as exc:
        row.update(valid=False,stop_reason=str(exc)); raise
    finally:
        try: clipped=clipped or ps.monitor_clipped()
        finally: all_off(); MANUAL_LOG.append(row)
    if clipped:
        row.update(valid=False,stop_reason='OUT2 clipped'); raise RuntimeError('OUT2 clipped')
    p=np.maximum(np.abs(np.asarray(z))**2-_background_on(grid),1e-12)
    ft=fit_lorentzian(grid,p,allow_second=True)
    if ft['resid']>SEARCH_FIT_MAX_RESID:
        row.update(valid=False,stop_reason='fit residual'); raise RuntimeError('replay fit rejected')
    measured_k=1.0-ft['lw']/LW
    row.update(valid=True,measured_k=float(measured_k),Q=float(ft['Q']),
               linewidth_hz=float(ft['lw']),residual=float(ft['resid']))
    print('MEASURED Q %.1f, linewidth %.2f kHz, K %.4f (requested %.4f)' %
          (ft['Q'],ft['lw']/1e3,measured_k,row['target_k']))
    if abs(measured_k-row['target_k'])>0.10:
        print('WARNING: measured K differs by >0.10; K/A is NOT silently changed')
    return row

while True:
    all_off()
    command=input('K, +step, -step, repeat, status, or quit [hardware OFF]: ').strip().lower()
    if command in ('q','quit','off'): break
    if command=='status':
        print('f0 %.6f MHz Q0 %.1f phase %.1f K/A %.5f last %s' %
              (F0/1e6,Q_SWEEP,PHI_NARROW,K_PER_A,current)); continue
    if command=='repeat':
        if current is None: print('no previous K'); continue
    elif command=='+step': current=(0.0 if current is None else current)+MANUAL_K_STEP
    elif command=='-step': current=(0.0 if current is None else current)-MANUAL_K_STEP
    else:
        try: current=float(command)
        except ValueError: print('unrecognised command'); continue
    try: replay_k(current)
    except Exception as exc: print('ABORTED: %s; hardware safe'%exc)
all_off()


---
# Emergency stop and summary

> ### âœ… PUMP ON
> Estimated bench time: **under 10 seconds**. Restart the pump, then run this cell. It is safe to run repeatedly or immediately in an emergency.


In [ ]:
pump('on', 'FINAL - emergency shutdown and summary', 0.1)
all_off()
print('OPEN LOOP: f0 %.6f MHz, linewidth %.2f kHz, Q %.1f' % (F0/1e6,LW/1e3,Q_SWEEP))
print('FEEDBACK: narrowing phase %.1f deg, K/A %.5f, manual limit %.4f' %
      (PHI_NARROW,K_PER_A,K_REACHABLE))
for row in globals().get('MANUAL_LOG',[]):
    print(row)
print('generator OFF; feedback OFF; OUT2 OFF; emulator OFF; acquisition OFF')
